In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import joblib
import os

print("All libraries imported!")


All libraries imported!


In [2]:
df = pd.read_csv("D:\\agro-agent\\datasets\\Crop_recommendation.csv")
print("Crop dataset loaded!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Crop dataset loaded!
Shape: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice


In [3]:
X = df[['N','P','K','temperature','humidity','ph','rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

crop_model = RandomForestClassifier(n_estimators=100, random_state=42)
crop_model.fit(X_train, y_train)

y_pred = crop_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Crop Model Accuracy:", round(acc*100, 2), "%")
print(classification_report(y_test, y_pred))

os.makedirs('../models', exist_ok=True)
joblib.dump(crop_model, '../models/crop_model.pkl')
print("Crop model saved to models/crop_model.pkl")

Crop Model Accuracy: 99.55 %
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        2

In [4]:
df2 = pd.read_csv("D:\\agro-agent\\datasets\\Fertilizer Prediction.csv")
print("Fertilizer Columns:", df2.columns.tolist())
print(df2.head(3))

Fertilizer Columns: ['Temparature', 'Humidity ', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name']
   Temparature  Humidity   Moisture Soil Type  Crop Type  Nitrogen  Potassium  \
0           26         52        38     Sandy      Maize        37          0   
1           29         52        45     Loamy  Sugarcane        12          0   
2           34         65        62     Black     Cotton         7          9   

   Phosphorous Fertilizer Name  
0            0            Urea  
1           36             DAP  
2           30        14-35-14  


In [5]:
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

df2['Soil Type']        = le_soil.fit_transform(df2['Soil Type'])
df2['Crop Type']        = le_crop.fit_transform(df2['Crop Type'])
df2['Fertilizer Name']  = le_fert.fit_transform(df2['Fertilizer Name'])

X2 = df2.drop('Fertilizer Name', axis=1)
y2 = df2['Fertilizer Name']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

fert_model = XGBClassifier(n_estimators=100, random_state=42)
fert_model.fit(X2_train, y2_train)

acc2 = accuracy_score(y2_test, fert_model.predict(X2_test))
print("Fertilizer Model Accuracy:", round(acc2*100, 2), "%")

joblib.dump(fert_model, '../models/fert_model.pkl')
joblib.dump(le_soil,    '../models/le_soil.pkl')
joblib.dump(le_crop,    '../models/le_crop.pkl')
joblib.dump(le_fert,    '../models/le_fert.pkl')
print("Fertilizer model saved!")

Fertilizer Model Accuracy: 100.0 %
Fertilizer model saved!


In [9]:
yield2 = pd.read_csv("D:\\agro-agent\\datasets\\yield_df.csv")
print("=== yield_df.csv ===")
print("Columns:", yield2.columns.tolist())
print("Shape:", yield2.shape)
print(yield2.head(3))

=== yield_df.csv ===
Columns: ['Unnamed: 0', 'Area', 'Item', 'Year', 'hg/ha_yield', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
Shape: (28242, 8)
   Unnamed: 0     Area         Item  Year  hg/ha_yield  \
0           0  Albania        Maize  1990        36613   
1           1  Albania     Potatoes  1990        66667   
2           2  Albania  Rice, paddy  1990        23333   

   average_rain_fall_mm_per_year  pesticides_tonnes  avg_temp  
0                         1485.0              121.0     16.37  
1                         1485.0              121.0     16.37  
2                         1485.0              121.0     16.37  


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Use yield_df which has all features combined
df_yield = yield2.drop('Unnamed: 0', axis=1)

# Encode categorical columns
le_area = LabelEncoder()
le_item = LabelEncoder()

df_yield['Area'] = le_area.fit_transform(df_yield['Area'])
df_yield['Item'] = le_item.fit_transform(df_yield['Item'])

# Drop rows with missing values
df_yield = df_yield.dropna()

X3 = df_yield[['Area', 'Item', 'Year', 
                'average_rain_fall_mm_per_year', 
                'pesticides_tonnes', 'avg_temp']]
y3 = df_yield['hg/ha_yield']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

yield_model = RandomForestRegressor(n_estimators=100, random_state=42)
yield_model.fit(X3_train, y3_train)

y3_pred = yield_model.predict(X3_test)
r2  = r2_score(y3_test, y3_pred)
mse = mean_squared_error(y3_test, y3_pred)

print("Yield Model R2 Score:", round(r2, 4))
print("Yield Model RMSE:", round(mse**0.5, 2))

joblib.dump(yield_model, '../models/yield_model.pkl')
joblib.dump(le_area,     '../models/le_area.pkl')
joblib.dump(le_item,     '../models/le_item.pkl')
print("Yield model saved!")

Yield Model R2 Score: 0.9857
Yield Model RMSE: 10181.76
Yield model saved!


In [12]:
!pip install matplotlib

  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl (8.1 MB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl (73 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     - -------------------------------------- 0.0/1.6 MB 281.8 kB/s eta 0:00:06
     - -------------------------------------- 0.1/1.6 MB 328.2 kB/s eta 0:00:05
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     --- ------------------------------------ 0.2/1.6 MB 416.


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Check dataset size first
print("Fertilizer dataset size:", df2.shape)
print("Unique fertilizers:", len(le_fert.classes_))
print("Fertilizer types:", le_fert.classes_)

# Detailed report
y2_pred = fert_model.predict(X2_test)
print("\nDetailed Classification Report:")
print(classification_report(y2_test, y2_pred))

# Check training vs testing accuracy
train_acc = accuracy_score(y2_train, fert_model.predict(X2_train))
test_acc  = accuracy_score(y2_test,  fert_model.predict(X2_test))

print("Training Accuracy:", round(train_acc*100, 2), "%")
print("Testing Accuracy: ", round(test_acc*100,  2), "%")

if train_acc == test_acc:
    print("\n✅ NOT overfit — train and test are equal")
else:
    print("\n⚠️ Possible overfit — train higher than test")

Fertilizer dataset size: (99, 9)
Unique fertilizers: 7
Fertilizer types: ['10-26-26' '14-35-14' '17-17-17' '20-20' '28-28' 'DAP' 'Urea']

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         1
           4       1.00      1.00      1.00         5
           5       1.00      1.00      1.00         5
           6       1.00      1.00      1.00         6

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

Training Accuracy: 100.0 %
Testing Accuracy:  100.0 %

✅ NOT overfit — train and test are equal


In [15]:
import os
models = os.listdir('../models/')
print("✅ Saved models:")
for m in models:
    print(" -", m)

✅ Saved models:
 - crop_model.pkl
 - fert_model.pkl
 - le_area.pkl
 - le_crop.pkl
 - le_fert.pkl
 - le_item.pkl
 - le_soil.pkl
 - yield_model.pkl


In [3]:
import joblib

le_crop = joblib.load("D:\\agro-agent\\models\\le_crop.pkl")
le_item = joblib.load("D:\\agro-agent\\models\\le_item.pkl")
le_area = joblib.load("D:\\agro-agent\\models\\le_area.pkl")

print("=== le_crop classes ===")
print(le_crop.classes_)

print("\n=== le_item classes ===")
print(le_item.classes_)

print("\n=== le_area classes ===")
print(le_area.classes_)

=== le_crop classes ===
['Barley' 'Cotton' 'Ground Nuts' 'Maize' 'Millets' 'Oil seeds' 'Paddy'
 'Pulses' 'Sugarcane' 'Tobacco' 'Wheat']

=== le_item classes ===
['Cassava' 'Maize' 'Plantains and others' 'Potatoes' 'Rice, paddy'
 'Sorghum' 'Soybeans' 'Sweet potatoes' 'Wheat' 'Yams']

=== le_area classes ===
['Albania' 'Algeria' 'Angola' 'Argentina' 'Armenia' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Belarus' 'Belgium'
 'Botswana' 'Brazil' 'Bulgaria' 'Burkina Faso' 'Burundi' 'Cameroon'
 'Canada' 'Central African Republic' 'Chile' 'Colombia' 'Croatia'
 'Denmark' 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador' 'Eritrea'
 'Estonia' 'Finland' 'France' 'Germany' 'Ghana' 'Greece' 'Guatemala'
 'Guinea' 'Guyana' 'Haiti' 'Honduras' 'Hungary' 'India' 'Indonesia' 'Iraq'
 'Ireland' 'Italy' 'Jamaica' 'Japan' 'Kazakhstan' 'Kenya' 'Latvia'
 'Lebanon' 'Lesotho' 'Libya' 'Lithuania' 'Madagascar' 'Malawi' 'Malaysia'
 'Mali' 'Mauritania' 'Mauritius' 'Mexico' 'Montenegro' 'Mo